# Transkribus AI combinations (OpenAI alone as of right now)

In [1]:
import difflib
import json
from pathlib import Path
from typing import Callable
import Levenshtein
from anyio.streams import file
from jiwer import cer, wer
import openai
import pandas as pd
import plotly.graph_objects as go
import kaleido

# files

In [2]:
with open("data/keys/tokens.json", "r") as f:
    tokens_dict = json.load(f)

# global variables

In [3]:
transkribus_corrected_folder = Path("data/texts/transkribus_corrected")

transkribus_inferenced_folder = Path("data/texts/transkribus_inferenced")
anno_original_folder = Path("data/texts/anno")

tsk_openai_zero_shot_folder = Path("data/texts/tsk_openai_zero_shot")
anno_openai_zero_shot_folder = Path("data/texts/anno_openai_zero_shot")

tsk_openai_one_shot_folder = Path("data/texts/tsk_openai_one_shot")
anno_openai_one_shot_folder = Path("data/texts/anno_openai_one_shot")

tsk_openai_two_shot_folder = Path("data/texts/tsk_openai_two_shot")
anno_openai_two_shot_folder = Path("data/texts/anno_openai_two_shot")

tsk_openai_three_shot_folder = Path("data/texts/tsk_openai_three_shot")
anno_openai_three_shot_folder = Path("data/texts/anno_openai_three_shot")

#api keys
openai_client = openai.OpenAI(api_key=tokens_dict["openai_token"])

# helper functions:

In [6]:
def clean(text:str) -> str:
    text = text.replace('-', '=').replace('¬', '=')
    return text

In [7]:
def clean_for_comp(text:str) -> str:
    text = text.replace('=  ', '=')
    text = text.replace('=\n', '')
    text = text.replace('\n', ' ').replace('= ', '').replace('    ', ' ').replace('   ', ' ')
    return text

In [8]:
def read_corrected_text(file_id: str) -> str:
    with open(transkribus_corrected_folder / Path(file_id + ".txt"), "r", encoding="utf-8") as f:
        return f.read()

In [9]:
def read_inferred_text(file_id: str) -> str:
    with open(transkribus_inferenced_folder / Path(file_id + ".txt"), "r", encoding="utf-8") as f:
        text = f.read()
        text = clean(text)
        return text

In [10]:
def read_anno_text(file_id: str) -> str:
    with open(anno_original_folder / Path(file_id + ".txt"), "r", encoding="utf-8") as f:
        text = f.read()
        text = clean(text)
        return text

In [11]:
def get_file_ids() -> list[str]:
    file_id_list = []
    for file_path in sorted(transkribus_corrected_folder.iterdir()):
        if "test" not in str(file_path):
            file_id_list.append(file_path.stem)
    return file_id_list

# prompts

In [12]:
def get_zero_shot_prompt(file) -> str:
    prompt = f"""
    Bitte korrigiere den folgenden Text, der eine fehlerhafte Transkription einer Zeitung aus dem frühen 20. Jahrhundert darstellt. Achte dabei auf folgende Regeln:
    
    ACHTUNG! Löse Zeilen auf, die eindeutig in die nächste Spalte überlaufen, oft gekennzeichnet mit "|" , indem du den Text trennst. Die zweite Hälfte muss an der passenden Stelle eingefügt werden. Achte auf den KONTEXT, die passende Stelle kann viele Zeilen später sein!
    
    Ersetze UNBEDINGT Worttrennungen am Zeilenende mit „Wort=\nbruch“, wenn Wörter in der Vorlage durch Zeilenumbruch getrennt wurden. Achte explizit darauf, SONST Zeilenumbrüche beizubehalten!
    
    Markiere unlesbare oder fehlende Textstellen mit „<...>“.
    
    Erhalte die ursprüngliche Orthographie und Interpunktion, abgesehen von den oben genannten Modifikationen.
    
    Achte explizit darauf, sonst ALLE Zeilenumbrüche im Text beizubehalten!
    
    Achte explizit darauf, AUSSCHLIEßLICH den korrigierten Text zurückzugeben, ohne weitere Erklärungen oder Kommentare.
    
    Hier ist der zu korrigierende Text:

    {file}
    """
    return prompt

In [13]:
def get_one_shot_prompt(file, comparison_file_one, correction_one) -> str:
    prompt = f"""
    Bitte korrigiere den folgenden Text, der eine fehlerhafte Transkription einer Zeitung aus dem frühen 20. Jahrhundert darstellt.
    
    Achte dabei auf folgende Regeln:
    
    Ersetze UNBEDINGT Worttrennungen am Zeilenende mit „Wort=\nbruch“, wenn Wörter in der Vorlage durch Zeilenumbruch getrennt wurden. Achte explizit darauf, SONST Zeilenumbrüche beizubehalten!
    
    Markiere unlesbare oder fehlende Textstellen mit „<...>“.
    
    Erhalte die ursprüngliche Orthographie und Interpunktion, abgesehen von den oben genannten Modifikationen.
    
    Achte explizit darauf, sonst ALLE Zeilenumbrüche im Text beizubehalten!
    
    Achte explizit darauf, AUSSCHLIEßLICH den korrigierten Text zurückzugeben, ohne weitere Erklärungen oder Kommentare.
    
    Hier ist ein Beispieltext:
    {comparison_file_one}
    
    Hier seine Beispielverbesserung:
    {correction_one}
    
    Hier ist der zu korrigierende Text:
    {file}
    """
    return prompt

In [14]:
def get_two_shot_prompt(file, comparison_file_one, correction_one, comparison_file_two, correction_two) -> str:
    prompt = f"""
    Bitte korrigiere den folgenden Text, der eine fehlerhafte Transkription einer Zeitung aus dem frühen 20. Jahrhundert darstellt.
    
    Achte dabei auf folgende Regeln:
    
    Ersetze UNBEDINGT Worttrennungen am Zeilenende mit „Wort=\nbruch“, wenn Wörter in der Vorlage durch Zeilenumbruch getrennt wurden. Achte explizit darauf, SONST Zeilenumbrüche beizubehalten!
    
    Markiere unlesbare oder fehlende Textstellen mit „<...>“.
    
    Erhalte die ursprüngliche Orthographie und Interpunktion, abgesehen von den oben genannten Modifikationen.
    
    Achte explizit darauf, sonst ALLE Zeilenumbrüche im Text beizubehalten!
    
    Achte explizit darauf, AUSSCHLIEßLICH den korrigierten Text zurückzugeben, ohne weitere Erklärungen oder Kommentare.
    
    Hier ist der erste Beispieltext:
    {comparison_file_one}
    
    Hier die Beispielverbesserung des ersten Texts:
    {correction_one}
    
    Hier ist der zweite Beispieltext:
    {comparison_file_two}
    
    Hier die Beispielverbesserung des zweiten Texts:
    {correction_two}
    
    Hier ist der zu korrigierende Text:
    {file}
    """
    return prompt

In [15]:
def get_three_shot_prompt(file, comparison_file_one, correction_one, comparison_file_two, correction_two, comparison_file_three, correction_three) -> str:
    prompt = f"""
    Bitte korrigiere den folgenden Text, der eine fehlerhafte Transkription einer Zeitung aus dem frühen 20. Jahrhundert darstellt.
    
    Achte dabei auf folgende Regeln:
    
    Ersetze UNBEDINGT Worttrennungen am Zeilenende mit „Wort=\nbruch“, wenn Wörter in der Vorlage durch Zeilenumbruch getrennt wurden. Achte explizit darauf, SONST Zeilenumbrüche beizubehalten!
    
    Markiere unlesbare oder fehlende Textstellen mit „<...>“.
    
    Erhalte die ursprüngliche Orthographie und Interpunktion, abgesehen von den oben genannten Modifikationen.
    
    Achte explizit darauf, sonst ALLE Zeilenumbrüche im Text beizubehalten!
    
    Achte explizit darauf, AUSSCHLIEßLICH den korrigierten Text zurückzugeben, ohne weitere Erklärungen oder Kommentare.
    
    Hier ist der erste Beispieltext:
    {comparison_file_one}
    
    Hier die Beispielverbesserung des ersten Texts:
    {correction_one}
    
    Hier ist der zweite Beispieltext:
    {comparison_file_two}
    
    Hier die Beispielverbesserung des zweiten Texts:
    {correction_two}
    
    Hier ist der dritte Beispieltext:
    {comparison_file_three}
    
    Hier die Beispielverbesserung des dritten Texts:
    {correction_three}
    
    Hier ist der zu korrigierende Text:
    {file}
    """
    return prompt

# Corrections with AI

this does not is not a correction, it just follows the same logic

In [16]:
def transkribus_alone(file_id: str) -> str:
    with open(transkribus_inferenced_folder / Path(file_id + ".txt"), "r", encoding="utf-8") as f:
        return f.read()

In [17]:
def anno_alone(file_id: str) -> str:
    with open(anno_original_folder / Path(file_id + ".txt"), "r", encoding="utf-8") as f:
        return f.read()

# openAI corrections

In [18]:
def openai_correction_base(prompt: str) -> str:
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt,
                    },
                ],
            }
        ],
    )
    return response.choices[0].message.content


In [19]:
def openai_correction_zero_shot_tsk(file_id) -> str:
    file = read_inferred_text(file_id)
    prompt = get_zero_shot_prompt(file)
    return openai_correction_base(prompt)

In [20]:
def openai_correction_zero_shot_anno(file_id) -> str:
    file = read_anno_text(file_id)

    prompt = get_zero_shot_prompt(file)
    return openai_correction_base(prompt)

In [21]:
def openai_correction_one_shot_tsk(file_id, comparison_file_one_id) -> str:
    file = read_inferred_text(file_id)
    comparison_file_one = read_inferred_text(comparison_file_one_id)
    correction_one = read_corrected_text(comparison_file_one_id)
    
    prompt = get_one_shot_prompt(file, comparison_file_one, correction_one)
    return openai_correction_base(prompt)

In [22]:
def openai_correction_one_shot_anno(file_id, comparison_file_one_id) -> str:
    file = read_anno_text(file_id)
    comparison_file_one = read_anno_text(comparison_file_one_id)
    correction_one = read_corrected_text(comparison_file_one_id)
    
    prompt = get_one_shot_prompt(file, comparison_file_one, correction_one)
    return openai_correction_base(prompt)

In [23]:
def openai_correction_two_shot_tsk(file_id, comparison_file_one_id, comparison_file_two_id) -> str:
    file = read_inferred_text(file_id)
    comparison_file_one = read_inferred_text(comparison_file_one_id)
    correction_one = read_corrected_text(comparison_file_one_id)
    comparison_file_two = read_inferred_text(comparison_file_two_id)
    correction_two = read_corrected_text(comparison_file_two_id)
    
    prompt = get_two_shot_prompt(file,comparison_file_one, correction_one, comparison_file_two, correction_two)
    return openai_correction_base(prompt)

In [24]:
def openai_correction_two_shot_anno(file_id, comparison_file_one_id, comparison_file_two_id) -> str:
    file = read_anno_text(file_id)
    comparison_file_one = read_inferred_text(comparison_file_one_id)
    correction_one = read_corrected_text(comparison_file_one_id)
    comparison_file_two = read_inferred_text(comparison_file_two_id)
    correction_two = read_corrected_text(comparison_file_two_id)

    prompt = get_two_shot_prompt(file, comparison_file_one, correction_one, comparison_file_two, correction_two)
    return openai_correction_base(prompt)

In [25]:
def openai_correction_three_shot_tsk(file_id, comparison_file_one_id, comparison_file_two_id, comparison_file_three_id) -> str:
    file = read_inferred_text(file_id)
    comparison_file_one = read_inferred_text(comparison_file_one_id)
    correction_one = read_corrected_text(comparison_file_one_id)
    comparison_file_two = read_inferred_text(comparison_file_two_id)
    correction_two = read_corrected_text(comparison_file_two_id)
    comparison_file_three= read_inferred_text(comparison_file_three_id)
    correction_three = read_corrected_text(comparison_file_three_id)
    
    prompt = get_three_shot_prompt(file,comparison_file_one, correction_one, comparison_file_two, correction_two, comparison_file_three, correction_three)
    return openai_correction_base(prompt)

In [26]:
def openai_correction_three_shot_anno(file_id, comparison_file_one_id, comparison_file_two_id, comparison_file_three_id) -> str:
    file = read_anno_text(file_id)
    comparison_file_one = read_inferred_text(comparison_file_one_id)
    correction_one = read_corrected_text(comparison_file_one_id)
    comparison_file_two = read_inferred_text(comparison_file_two_id)
    correction_two = read_corrected_text(comparison_file_two_id)
    comparison_file_three= read_inferred_text(comparison_file_three_id)
    correction_three = read_corrected_text(comparison_file_three_id)
    
    prompt = get_three_shot_prompt(file,comparison_file_one, correction_one, comparison_file_two, correction_two, comparison_file_three, correction_three)
    return openai_correction_base(prompt)

# get all function

In [34]:
def get_all_correction_func() -> list[Callable]:
    return [
        transkribus_alone,
        anno_alone,
        openai_correction_zero_shot_tsk,
        openai_correction_one_shot_tsk,
        openai_correction_two_shot_tsk,
        openai_correction_three_shot_tsk,
        openai_correction_zero_shot_anno,
        openai_correction_one_shot_anno,
        openai_correction_two_shot_anno,
        openai_correction_three_shot_anno
    ]

In [31]:
def get_correction_output_folder(corr_func: Callable) -> str:
    if corr_func is transkribus_alone:
        return transkribus_inferenced_folder
    elif corr_func is anno_alone:
        return anno_original_folder
    elif corr_func is openai_correction_zero_shot_tsk:
        return tsk_openai_zero_shot_folder
    elif corr_func is openai_correction_zero_shot_anno:
        return anno_openai_zero_shot_folder
    elif corr_func is openai_correction_one_shot_tsk:
        return tsk_openai_one_shot_folder
    elif corr_func is openai_correction_one_shot_anno:
        return anno_openai_one_shot_folder
    elif corr_func is openai_correction_two_shot_tsk:
        return tsk_openai_two_shot_folder
    elif corr_func is openai_correction_two_shot_anno:
        return anno_openai_two_shot_folder
    elif corr_func is openai_correction_three_shot_tsk:
        return tsk_openai_three_shot_folder
    elif corr_func is openai_correction_three_shot_anno:
        return anno_openai_three_shot_folder

# correct all:

In [ ]:
# change this boolean to run the entire chain. Is disabled by default to avoid unncessary API costs.
enable_correct_all = False
if enable_correct_all:

    # iterate over functions
    for corr_func in get_all_correction_func():
        print(f"corr_func: {corr_func.__name__}")

        # iterate over images
        file_id_list = get_file_ids()
        for file_id in file_id_list:
            print(f"file_id: {file_id}")

            # do inference
            if corr_func is openai_correction_one_shot_tsk or corr_func is openai_correction_one_shot_anno:
                if file_id.endswith("1"):
                    comparison_file_one_id = file_id[:-1] + "2"
                elif file_id.endswith("2"):
                    comparison_file_one_id = file_id[:-1] + "3"
                elif file_id.endswith("3"):
                    comparison_file_one_id = file_id[:-1] + "2"
                elif file_id.endswith("4"):
                    comparison_file_one_id = file_id[:-1] + "3"
                elif file_id.endswith("5"):
                    comparison_file_one_id = file_id[:-1] + "2"
                elif file_id.endswith("6"):
                    comparison_file_one_id = file_id[:-1] + "1"
                elif file_id.endswith("7"):
                    comparison_file_one_id = file_id[:-1] + "3"
                corrected_text = corr_func(file_id, comparison_file_one_id)
                
            elif corr_func is openai_correction_two_shot_tsk or corr_func is openai_correction_two_shot_anno:
                if file_id.endswith("1"):
                    comparison_file_one_id = file_id[:-1] + "2"
                    comparison_file_two_id = file_id[:-1] + "3"
                elif file_id.endswith("2"):
                    comparison_file_one_id = file_id[:-1] + "3"
                    comparison_file_two_id = file_id[:-1] + "4"
                elif file_id.endswith("3"):
                    comparison_file_one_id = file_id[:-1] + "4"
                    comparison_file_two_id = file_id[:-1] + "2"
                elif file_id.endswith("4"):
                    comparison_file_one_id = file_id[:-1] + "3"
                    comparison_file_two_id = file_id[:-1] + "1"
                elif file_id.endswith("5"):
                    comparison_file_one_id = file_id[:-1] + "2"
                    comparison_file_two_id = file_id[:-1] + "3"
                elif file_id.endswith("6"):
                    comparison_file_one_id = file_id[:-1] + "1"
                    comparison_file_two_id = file_id[:-1] + "3"
                elif file_id.endswith("7"):
                    comparison_file_one_id = file_id[:-1] + "3"
                    comparison_file_two_id = file_id[:-1] + "4"
                corrected_text = corr_func(file_id, comparison_file_one_id, comparison_file_two_id) 
                
            elif corr_func is openai_correction_three_shot_tsk or corr_func is openai_correction_three_shot_anno:
                if file_id.endswith("1"):
                    comparison_file_one_id = file_id[:-1] + "2"
                    comparison_file_two_id = file_id[:-1] + "3"
                    comparison_file_three_id = file_id[:-1] + "4"
                elif file_id.endswith("2"):
                    comparison_file_one_id = file_id[:-1] + "3"
                    comparison_file_two_id = file_id[:-1] + "4"
                    comparison_file_three_id = file_id[:-1] + "1"
                elif file_id.endswith("3"):
                    comparison_file_one_id = file_id[:-1] + "4"
                    comparison_file_two_id = file_id[:-1] + "2"
                    comparison_file_three_id = file_id[:-1] + "1"
                elif file_id.endswith("4"):
                    comparison_file_one_id = file_id[:-1] + "3"
                    comparison_file_two_id = file_id[:-1] + "1"
                    comparison_file_three_id = file_id[:-1] + "2"
                elif file_id.endswith("5"):
                    comparison_file_one_id = file_id[:-1] + "2"
                    comparison_file_two_id = file_id[:-1] + "3"
                    comparison_file_three_id = file_id[:-1] + "4"
                elif file_id.endswith("6"):
                    comparison_file_one_id = file_id[:-1] + "1"
                    comparison_file_two_id = file_id[:-1] + "3"
                    comparison_file_three_id = file_id[:-1] + "4"
                elif file_id.endswith("7"):
                    comparison_file_one_id = file_id[:-1] + "3"
                    comparison_file_two_id = file_id[:-1] + "4"
                    comparison_file_three_id = file_id[:-1] + "2"
                corrected_text = corr_func(file_id, comparison_file_one_id, comparison_file_two_id, comparison_file_three_id)
            
            else:
                corrected_text = corr_func(file_id)

            # write inferenced text
            ocr_output_folder = get_correction_output_folder(corr_func)
            if ocr_output_folder:
                with open(ocr_output_folder / Path(file_id + ".txt"), "w", encoding="utf-8") as f:
                    f.write(corrected_text)